# Clustering of Countries based on Socio-Economic Indicators

This project aims to categorize countries based on socio-economic and health indicators using clustering techniques. The goal is to identify countries that are in the greatest need of financial aid so that the NGO can allocate resources effectively.

## Import Required Libraries

In this step, we import the necessary Python libraries used for data manipulation, visualization, and machine learning.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

## Load Dataset

The dataset contains socio-economic and health indicators for 167 countries. These indicators include GDP per capita, income levels, child mortality rates, life expectancy, and others that reflect the development level of each country.

In [ ]:
df = pd.read_csv("Country-data.csv")

df.head()
df.info()
df.describe()

## Data Preparation

The 'country' column contains categorical data (country names). Since clustering algorithms require numerical data, this column is removed before applying clustering techniques.

In [ ]:
df1 = df.drop('country', axis=1)
df.shape

## Exploratory Data Analysis

EDA is performed to understand the distribution of variables and identify patterns in the dataset. Histograms help visualize the distribution of each variable.

From the histogram plots, we observe that variables such as **GDP per capita (gdpp)** and **income** are right-skewed, meaning a small number of countries have very high economic values while the majority of countries have moderate or low values. This reflects real-world economic inequality among countries.

In [ ]:
df1.hist(figsize=(12,10))
plt.show()

## Outlier Analysis

Boxplots are used to detect extreme values in the dataset. Although several variables contain outliers, these represent real economic differences between countries, so they are retained for the clustering analysis.

In [ ]:
plt.figure(figsize=(15,8))
sns.boxplot(data=df1)
plt.xticks(rotation=90)
plt.show()

## Feature Scaling

Clustering algorithms rely on distance calculations. Since the variables in the dataset have different ranges (for example, GDP can reach very large values compared to other features), feature scaling is applied using StandardScaler so that all variables contribute equally to the clustering process.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

df_scaled = scaler.fit_transform(df1)

df_scaled = pd.DataFrame(df_scaled, columns=df1.columns)

df_scaled.head()

## Determining Optimal Number of Clusters

The Elbow Method is used to determine the optimal number of clusters for K-Means clustering. The method plots the Within-Cluster Sum of Squares (WCSS) against the number of clusters. The point where the decrease in WCSS slows down significantly represents the optimal number of clusters.

In [ ]:
from sklearn.cluster import KMeans

wcss = []

for i in range(1, 11):
    kmeans = KMeans(n_clusters=i, random_state=42, n_init=10)
    kmeans.fit(df_scaled)
    wcss.append(kmeans.inertia_)

plt.figure(figsize=(8,5))
plt.plot(range(1,11), wcss, marker='o')
plt.title("Elbow Method")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("WCSS")
plt.show()

## K-Means Clustering

Based on the Elbow Method, the optimal number of clusters is selected as 3. The K-Means algorithm is applied to group countries based on their socio-economic characteristics.

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)

kmeans.fit(df_scaled)

clusters = kmeans.labels_

## Assign Cluster Labels

The cluster labels generated by the K-Means algorithm are added to the original dataset to identify which cluster each country belongs to.

In [ ]:
df['cluster'] = clusters

df['cluster'].value_counts()

## Cluster Interpretation

The clusters are analyzed using key indicators such as GDP per capita, income, and child mortality rate. These indicators help distinguish between developed, developing, and underdeveloped countries.

In [ ]:
df.groupby('cluster')[['gdpp','income','child_mort']].mean().round(2)

The cluster analysis shows that:

Cluster 0 represents developing countries with moderate economic indicators.
Cluster 1 represents developed countries with high income levels and low child mortality.
Cluster 2 represents underdeveloped countries with low income and GDP levels along with high child mortality rates.

Countries in **Cluster 2** have the lowest economic development indicators and the highest child mortality rates, indicating poor economic conditions and limited healthcare resources. Therefore, these countries require the most financial assistance and should be prioritized by the NGO.

## Cluster Visualization

A scatter plot is used to visualize the relationship between GDP per capita and child mortality rate across the clusters.

In [ ]:
plt.figure(figsize=(8,6))
sns.scatterplot(x='gdpp', y='child_mort', hue='cluster', data=df, palette='Set1')
plt.title("Country Clusters based on GDP and Child Mortality")
plt.show()

## Visualization: Income vs GDP

This scatter plot visualizes the relationship between a country's income and GDP per capita across different clusters. Countries belonging to the same cluster share similar socio-economic characteristics. This plot helps in understanding how income levels correspond to economic output and how countries are grouped based on development indicators.

In [ ]:
plt.figure(figsize=(8,6))
sns.scatterplot(x='income', y='gdpp', hue='cluster', data=df, palette='Set1')
plt.title("Country Clusters based on Income and GDP")
plt.show()

## Visualization: Income vs Child Mortality

This scatter plot shows the relationship between income levels and child mortality rates for each country. Generally, countries with lower income levels tend to have higher child mortality rates. The clustering helps identify groups of countries with similar development patterns, highlighting the countries that require the most aid.

In [ ]:
plt.figure(figsize=(8,6))
sns.scatterplot(x='income', y='child_mort', hue='cluster', data=df, palette='Set1')
plt.title("Country Clusters based on Income and Child Mortality")
plt.show()

## Hierarchical Clustering

Hierarchical clustering is performed to validate the results obtained from the K-Means clustering method. A dendrogram is used to visualize how clusters are formed.

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage

plt.figure(figsize=(10,6))

linked = linkage(df_scaled, method='ward')

dendrogram(linked)

plt.title("Dendrogram")
plt.show()

## Identifying Countries Needing Aid

In [ ]:
poor_countries = df[df['cluster'] == 2]

poor_countries[['country','gdpp','income','child_mort']].sort_values(by='gdpp').head(10)

### Final Countries Recommended for Aid

Based on the clustering results and economic indicators, the following countries have the lowest GDP per capita and income levels along with high child mortality rates. These countries should be prioritized for financial assistance.

In [ ]:
top5_countries = poor_countries[['country','gdpp','income','child_mort']] \
                .sort_values(by='gdpp') \
                .head(5)

top5_countries

## Conclusion

The clustering analysis identified three distinct groups of countries based on socio-economic indicators. One cluster contains countries with high GDP and income levels, representing developed nations. Another cluster contains moderately developed countries. The third cluster contains countries with very low income and GDP levels along with high child mortality rates. These countries should be prioritized for financial aid by the NGO.